# 03 · Prediction sample and features

Builds the sample for predicting **no recorded blood pressure measurement within 90 days of an elevated reading**, the alternative 30- to 365-day targets, and the features measured at the prediction date.

**Reads (from notebook 01, in `DATA_DIR`):** `eligibility`, `covariates`

**Produces (patient-level, `DATA_DIR`):** `pred_cohort`, `pred_features`

**Produces (aggregate, `results/`):** `03_prediction_counts.json`

**Paper items:** Methods, *Prediction of no recorded blood pressure measurement*; Supplementary Methods, *Prediction targets*; Figure 1A (prediction branch); Table S9 (prediction rows); missing-value rows of Table 1.

| Step | What it does |
|---|---|
| 1 | Prediction date: first systolic reading > 140 mmHg on or after index |
| 2 | Coverage, survival, and the target for each window (30, 90, 180, 270, 365 days) |
| 3 | Death as a competing event (Table S5B sample) |
| 4 | Features, using only information available at the prediction date |

## 0 · Setup

In [ ]:
# ---- edit these ----
REPO_DIR   = "/content/drive/MyDrive/Ronveer Sandhu/Project 1 -MIMIC analysis-hypertension retention/htn-bp-followup-mimic"
MIMIC_HOSP = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/hosp"
MIMIC_ED   = "/content/drive/MyDrive/MIMIC Data/mimic-iv-2.2/ed"
DATA_DIR   = "/content/drive/MyDrive/htn_bp_data"

# prediction date: "any reading" = first day with any systolic reading > 140 (paper's definition)
#                  "daily mean"  = first day whose average systolic reading is > 140
PRED_RULE = "any reading"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install duckdb==1.3.2

import os, sys
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
from htn_utils import *

import numpy as np
import pandas as pd

RESULTS_DIR = os.path.join(REPO_DIR, "results")
WINDOWS = [30, 90, 180, 270, 365]

el  = load_df(DATA_DIR, "eligibility")
cov = load_df(DATA_DIR, "covariates")
el["index_date"] = pd.to_datetime(el.index_date)

## 1 · Prediction date

Same BP and encounter definitions as notebook 01. The prediction date is each patient's first day on or after index with a systolic reading above 140 mmHg. Diastolic pressure does not start the window.

In [ ]:
con = connect(MIMIC_HOSP, MIMIC_ED)

con.execute('''
CREATE OR REPLACE TABLE bp AS
SELECT * FROM (
  SELECT subject_id, CAST(chartdate AS DATE) AS bp_date,
         TRY_CAST(SPLIT_PART(result_value, '/', 1) AS INTEGER) AS sbp,
         TRY_CAST(SPLIT_PART(result_value, '/', 2) AS INTEGER) AS dbp
  FROM omr WHERE result_name LIKE 'Blood Pressure%'
) WHERE sbp BETWEEN 60 AND 300 AND dbp BETWEEN 30 AND 200''')

con.execute('''
CREATE OR REPLACE TABLE encounters AS
WITH adm AS (SELECT subject_id, CAST(admittime AS DATE) AS enc_date, 1 AS rank FROM admissions),
     edv AS (SELECT subject_id, CAST(intime AS DATE) AS enc_date, 2 AS rank
             FROM ed_edstays WHERE hadm_id IS NULL),
     opt AS (SELECT DISTINCT subject_id, CAST(chartdate AS DATE) AS enc_date, 3 AS rank FROM omr),
     u   AS (SELECT * FROM adm UNION ALL SELECT * FROM edv UNION ALL SELECT * FROM opt)
SELECT subject_id, enc_date,
       CASE MIN(rank) WHEN 1 THEN 'inpatient' WHEN 2 THEN 'ed' ELSE 'outpatient' END AS enc_type
FROM u GROUP BY subject_id, enc_date''')

pid = el[["subject_id", "index_date"]].copy()
pid["index_date"] = pid.index_date.dt.date
con.register("pid", pid)

# BP readings on or after index, as days from index
bpd = con.execute('''
  SELECT b.subject_id, DATE_DIFF('day', CAST(p.index_date AS DATE), b.bp_date) AS dd, b.sbp, b.dbp
  FROM bp b JOIN pid p ON p.subject_id = b.subject_id
  WHERE b.bp_date >= CAST(p.index_date AS DATE)''').df()

d_any  = bpd[bpd.sbp > 140].groupby("subject_id").dd.min()
dm     = bpd.groupby(["subject_id", "dd"]).sbp.mean().reset_index()
d_mean = dm[dm.sbp > 140].groupby("subject_id").dd.min()
print(f"patients with a qualifying day:  any reading {len(d_any):,} | daily mean {len(d_mean):,}")

idx_map = d_any if PRED_RULE == "any reading" else d_mean
P = el[["subject_id", "index_date", "admin_day", "days_to_death"]].copy()
P["idx_day"] = P.subject_id.map(idx_map)
P = P[P.idx_day.notna()].copy()
P["idx_day"] = P.idx_day.astype(int)
P["idx_date"] = (P.index_date + pd.to_timedelta(P.idx_day, unit="D")).dt.date

check("First systolic reading > 140 mmHg", len(P), 32101)

# next BP date after the prediction date
bpdays = bpd[["subject_id", "dd"]].drop_duplicates().merge(P[["subject_id", "idx_day"]], on="subject_id")
P["next_bp_day"] = P.subject_id.map(bpdays[bpdays.dd > bpdays.idx_day].groupby("subject_id").dd.min())

no_data_after = P.admin_day <= P.idx_day
check("  no data after the prediction date", int(no_data_after.sum()), 711)
P = P[~no_data_after].copy()
check("Data after the prediction date", len(P), 31390)

## 2 · Targets for each window

For a window of H days after the prediction date, a patient is in that target's sample when database coverage reaches day H and they are alive through day H. The target is **no BP measurement recorded within H days**. Later contact is not required.

The *deaths counted* variant (Table S5A) also keeps covered patients who died before any BP reading and counts them as outcomes.

In [ ]:
for H in WINDOWS:
    hi = P.idx_day + H
    P[f"covered_{H}"] = P.admin_day >= hi
    P[f"died_{H}"]    = P.days_to_death.notna() & (P.days_to_death <= hi)
    P[f"bp_{H}"]      = P.next_bp_day.notna() & (P.next_bp_day <= hi)
    P[f"target_{H}"]  = np.where(P[f"covered_{H}"] & ~P[f"died_{H}"], (~P[f"bp_{H}"]).astype(float), np.nan)

# deaths before any BP counted as outcomes (90 days)
death_before_bp = P.covered_90 & P.died_90 & ~P.bp_90
P["target_90_deaths"] = np.where(P.covered_90 & (~P.died_90 | death_before_bp),
                                 (~P.bp_90 | death_before_bp).astype(float), np.nan)

paper = {30: (31278, 18608), 90: (30942, 10899), 180: (30081, 6878), 270: (29127, 5172), 365: (27792, 4378)}
for H in WINDOWS:
    s = P[f"target_{H}"].dropna()
    print(f"--- {H}-day target ---")
    check(f"  data ended within {H} days", int((~P[f'covered_{H}']).sum()),
          {30: 78, 90: 254, 180: 559, 270: 1055, 365: 1971}[H])
    check(f"  died within {H} days", int((P[f'covered_{H}'] & P[f'died_{H}']).sum()),
          {30: 34, 90: 194, 180: 750, 270: 1208, 365: 1627}[H])
    check(f"  sample", len(s), paper[H][0])
    check(f"  no recorded BP", int(s.sum()), paper[H][1])

s = P.target_90_deaths.dropna()
print("--- 90-day target, deaths before BP counted ---")
check("  sample", len(s), 31016)
check("  outcome", int(s.sum()), 10973)

### Timing in the primary (90-day) sample

In [ ]:
S = P[P.target_90.notna()]
print(f"prediction date, days after index: median {S.idx_day.median():.0f}   (paper: 22)")
gap = (S.next_bp_day - S.idx_day).dropna()
q = gap.quantile([.25, .5, .75])
print(f"days to the next BP reading: median {q[.5]:.0f} (IQR {q[.25]:.0f}-{q[.75]:.0f})   (paper: 35, IQR 14-91)")

## 3 · Death as a competing event (Table S5B)

Among all 31,390 patients with data after the prediction date: time to the next BP reading, with death before any reading as a competing event.

A BP reading or death within 90 days counts as an event even if it falls after the patient's coverage end, because coverage is a conservative bound on the *hospital* data and outpatient readings continue into 2021. Patients with neither are censored at the end of coverage or day 90.

In [ ]:
end_ = np.minimum(P.admin_day.values - P.idx_day.values, 90)
t_bp = (P.next_bp_day - P.idx_day).values.astype(float)
t_d  = (P.days_to_death - P.idx_day).values.astype(float)
bp_first = ~np.isnan(t_bp) & (t_bp <= 90) & (np.isnan(t_d) | (t_bp <= t_d))
d_first  = ~bp_first & ~np.isnan(t_d) & (t_d <= 90)
cov_end  = ~bp_first & ~d_first & (end_ < 90)
P["cr90_status"] = np.select([bp_first, d_first], [1, 2], 0)
P["cr90_time"]   = np.where(bp_first, t_bp, np.where(d_first, t_d, end_)).clip(min=0.5)

check("BP reading within 90 days",   int(bp_first.sum()), 20311)
check("Death before any BP reading", int(d_first.sum()),  78)
check("Coverage ended first",        int(cov_end.sum()),  102)

## 4 · Features at the prediction date

All features use only information available at the prediction date:

- **Visit history** (encounters strictly before the prediction date): number of prior encounters, days since the last visit, average interval between visits, and regularity of intervals (normalized entropy). Patients with fewer than two prior encounters have no interval-based values.
- **Blood pressure history** (readings strictly before the prediction date, systolic 60–300): mean, standard deviation (variability), and linear trend per day.
- **Blood pressure level**: mean systolic and diastolic in the 365 days up to and including the prediction date.
- **Labs, body mass index, antihypertensive drug count**: 365 days up to and including the prediction date; eGFR by the 2021 CKD-EPI equation without race.
- **From notebook 01**: age, sex, race and ethnicity, insurance (Medicare, Medicaid, Other), language, marital status, prior emergency/inpatient/total encounters before index, and Elixhauser comorbidities from diagnoses up to index.

The final 62-feature list is set in notebook 04.

In [ ]:
pt = P[["subject_id", "idx_date"]].copy()
con.register("pt", pt)

# --- visit history before the prediction date ---
e = con.execute('''SELECT e.subject_id, CAST(e.enc_date AS DATE) AS d FROM encounters e
                   JOIN pt ON pt.subject_id = e.subject_id WHERE CAST(e.enc_date AS DATE) < pt.idx_date''').df()
e = e.merge(pt, on="subject_id").sort_values(["subject_id", "d"])
e["d"] = pd.to_datetime(e.d); e["idx_date"] = pd.to_datetime(e.idx_date)

def temporal_feats(g):
    dates = g.d.values; n = len(dates)
    if n < 2:
        return pd.Series({"n_prior_enc": n, "recency_days": np.nan, "interval_entropy": np.nan, "mean_interval": np.nan})
    gaps = np.diff(dates).astype("timedelta64[D]").astype(float)
    p = gaps / gaps.sum() if gaps.sum() > 0 else None
    ent = -(p * np.log(p + 1e-12)).sum() / np.log(len(gaps)) if p is not None and len(gaps) > 1 else 0.0
    return pd.Series({"n_prior_enc": n,
                      "recency_days": (g.idx_date.iloc[0] - dates[-1]) / np.timedelta64(1, "D"),
                      "interval_entropy": ent, "mean_interval": gaps.mean()})

tf = e.groupby("subject_id").apply(temporal_feats, include_groups=False).reset_index()
print(f"visit history: {len(tf):,} patients")

In [ ]:
# --- BP history before the prediction date (systolic 60-300 only) ---
bs = con.execute('''SELECT o.subject_id, CAST(o.chartdate AS DATE) AS d,
                    TRY_CAST(SPLIT_PART(o.result_value, '/', 1) AS INTEGER) AS sbp
                    FROM omr o JOIN pt ON pt.subject_id = o.subject_id
                    WHERE o.result_name LIKE 'Blood Pressure%' AND CAST(o.chartdate AS DATE) < pt.idx_date''').df()
bs = bs[bs.sbp.between(60, 300)].copy()
bs["d"] = pd.to_datetime(bs.d)

def bp_feats(g):
    s = g.sbp.values
    if len(s) < 2:
        return pd.Series({"sbp_mean": s.mean() if len(s) else np.nan, "sbp_sd": np.nan, "sbp_trend": np.nan})
    t = (g.d - g.d.min()).dt.days.values
    return pd.Series({"sbp_mean": s.mean(), "sbp_sd": s.std(),
                      "sbp_trend": np.polyfit(t, s, 1)[0] if np.ptp(t) > 0 else 0.0})

bf = bs.groupby("subject_id").apply(bp_feats, include_groups=False).reset_index()

# --- BP level in the 365 days up to and including the prediction date ---
b = con.execute('''SELECT b.subject_id, b.bp_date AS d, b.sbp, b.dbp FROM bp b
                   JOIN pt ON pt.subject_id = b.subject_id WHERE b.bp_date <= pt.idx_date''').df().merge(pt, on="subject_id")
b["d"] = pd.to_datetime(b.d); b["idx_date"] = pd.to_datetime(b.idx_date)
bpb = (b[b.d >= b.idx_date - pd.Timedelta(days=365)]
         .groupby("subject_id").agg(sbp_baseline=("sbp", "mean"), dbp_baseline=("dbp", "mean")).reset_index())
print(f"BP history: {len(bf):,} patients | BP level: {len(bpb):,} patients")

In [ ]:
# --- labs, BMI, antihypertensive drugs in [prediction - 365, prediction] ---
labs = con.execute('''
  SELECT pt.subject_id,
         AVG(l.valuenum) FILTER (WHERE l.itemid = 50912) AS creatinine,
         AVG(l.valuenum) FILTER (WHERE l.itemid = 50852) AS hba1c
  FROM pt JOIN labevents l ON l.subject_id = pt.subject_id
  WHERE l.itemid IN (50912, 50852) AND l.valuenum IS NOT NULL
    AND CAST(l.charttime AS DATE) BETWEEN pt.idx_date - 365 AND pt.idx_date
  GROUP BY pt.subject_id''').df()

bmi = con.execute('''
  SELECT pt.subject_id, AVG(TRY_CAST(o.result_value AS DOUBLE)) AS bmi
  FROM pt JOIN omr o ON o.subject_id = pt.subject_id
  WHERE o.result_name LIKE 'BMI%' AND CAST(o.chartdate AS DATE) BETWEEN pt.idx_date - 365 AND pt.idx_date
    AND TRY_CAST(o.result_value AS DOUBLE) BETWEEN 12 AND 80
  GROUP BY pt.subject_id''').df()

rxd = con.execute(f'''
  SELECT pt.subject_id, COUNT(DISTINCT LOWER(r.drug)) AS n_antihtn_drugs
  FROM pt JOIN prescriptions r ON r.subject_id = pt.subject_id
  WHERE {sql_contains('r.drug', ANTIHTN)}
    AND CAST(r.starttime AS DATE) BETWEEN pt.idx_date - 365 AND pt.idx_date
  GROUP BY pt.subject_id''').df()
print(f"labs: {len(labs):,} | BMI: {len(bmi):,} | drugs: {len(rxd):,}")

In [ ]:
# --- assemble ---
# notebook 01 covariates measured around index are replaced by the prediction-date versions above
cvp = cov.drop(columns=["sbp_baseline", "dbp_baseline", "creatinine", "egfr", "hba1c", "bmi",
                        "hba1c_missing", "bmi_missing", "n_antihtn_drugs", "index_date"], errors="ignore")
target_cols = [f"target_{H}" for H in WINDOWS] + ["target_90_deaths"]

F = (P[["subject_id", "idx_date", "idx_day"] + target_cols]
     .merge(cvp, on="subject_id", how="left").merge(tf, on="subject_id", how="left")
     .merge(bf, on="subject_id", how="left").merge(bpb, on="subject_id", how="left")
     .merge(labs, on="subject_id", how="left").merge(bmi, on="subject_id", how="left")
     .merge(rxd, on="subject_id", how="left"))
F["n_antihtn_drugs"] = F.n_antihtn_drugs.fillna(0)
F["n_prior_enc"]     = F.n_prior_enc.fillna(0)

k  = np.where(F.gender == "F", 0.7, 0.9)
a_ = np.where(F.gender == "F", -0.241, -0.302)
r_ = F.creatinine / k
F["egfr"] = 142 * np.minimum(r_, 1) ** a_ * np.maximum(r_, 1) ** -1.2 * 0.9938 ** F.age_index \
            * np.where(F.gender == "F", 1.012, 1.0)
F["hba1c_missing"] = F.hba1c.isna()
F["bmi_missing"]   = F.bmi.isna()
F["insurance"] = np.where(F.insurance.eq("Medicaid"), "Medicaid",
                          np.where(F.insurance.eq("Medicare"), "Medicare", "Other"))
print(f"feature frame: {F.shape}")

### Checks against Table 1 (90-day sample)

In [ ]:
S = F[F.target_90.notna()]
check("Prediction sample",                         len(S), 30942)
check("No recorded BP within 90 days",             int(S.target_90.sum()), 10899)
check("Missing: days since last visit",            int(S.recency_days.isna().sum()), 3200)
check("Missing: SBP variability (< 2 readings)",   int(S.sbp_sd.isna().sum()), 9668)
# Table 1's "earlier readings" row (5,928 with none) also requires diastolic 30-200 and is built in notebook 08;
# the model feature below keeps every systolic reading of 60-300.
print(f"No systolic readings before prediction (feature): {int(S.sbp_mean.isna().sum()):,}")
check("Missing: eGFR",                             int(S.egfr.isna().sum()), 1711)
check("Missing: HbA1c",                            int(S.hba1c.isna().sum()), 18222)
check("Missing: BMI",                              int(S.bmi.isna().sum()), 5942)
q = S.n_prior_enc.quantile([.25, .5, .75])
print(f"prior encounters, median (IQR): {q[.5]:.0f} ({q[.25]:.0f}-{q[.75]:.0f})   (paper: 6, IQR 3-13)")
q = S.recency_days.quantile([.25, .5, .75])
print(f"days since last visit, median (IQR): {q[.5]:.0f} ({q[.25]:.0f}-{q[.75]:.0f})   (paper: 30, IQR 13-82)")

## 5 · Summary of all checks

In [ ]:
results = []
def chk(label, value, expected):
    results.append((value == expected, label, value, expected))

print(f"Prediction rule: {PRED_RULE}  (any reading: {len(d_any):,} | daily mean: {len(d_mean):,})\n")

chk("First systolic reading > 140",   len(P) + int(no_data_after.sum()), 32101)
chk("No data after prediction date",  int(no_data_after.sum()), 711)
chk("Data after prediction date",     len(P), 31390)

exp = {30: (78, 34, 31278, 18608), 90: (254, 194, 30942, 10899), 180: (559, 750, 30081, 6878),
       270: (1055, 1208, 29127, 5172), 365: (1971, 1627, 27792, 4378)}
for H in WINDOWS:
    s = P[f"target_{H}"].dropna()
    e1, e2, e3, e4 = exp[H]
    chk(f"{H:>3}d: data ended",     int((~P[f'covered_{H}']).sum()), e1)
    chk(f"{H:>3}d: died",           int((P[f'covered_{H}'] & P[f'died_{H}']).sum()), e2)
    chk(f"{H:>3}d: sample",         len(s), e3)
    chk(f"{H:>3}d: no recorded BP", int(s.sum()), e4)
s = P.target_90_deaths.dropna()
chk(" 90d deaths counted: sample",  len(s), 31016)
chk(" 90d deaths counted: outcome", int(s.sum()), 10973)

chk("Competing: BP within 90 days",    int(bp_first.sum()), 20311)
chk("Competing: death before any BP",  int(d_first.sum()), 78)
chk("Competing: coverage ended first", int(cov_end.sum()), 102)

S = F[F.target_90.notna()]
chk("Missing: days since last visit", int(S.recency_days.isna().sum()), 3200)
chk("Missing: SBP variability",       int(S.sbp_sd.isna().sum()), 9668)
chk("Missing: eGFR",                  int(S.egfr.isna().sum()), 1711)
chk("Missing: HbA1c",                 int(S.hba1c.isna().sum()), 18222)
chk("Missing: BMI",                   int(S.bmi.isna().sum()), 5942)

for ok, label, v, e in results:
    print(f"{'OK      ' if ok else 'MISMATCH'}  {label:38s} {v:>8,}   (paper {e:,})")
print(f"\n{sum(r[0] for r in results)} of {len(results)} checks match")

S90 = P[P.target_90.notna()]
g  = (S90.next_bp_day - S90.idx_day).dropna().quantile([.25, .5, .75])
pe = S.n_prior_enc.quantile([.25, .5, .75]); rd = S.recency_days.quantile([.25, .5, .75])
print(f"\nMedian days index -> prediction: {S90.idx_day.median():.0f}   [22]")
print(f"Median days to next BP:          {g[.5]:.0f} ({g[.25]:.0f}-{g[.75]:.0f})   [35 (14-91)]")
print(f"Prior encounters:                {pe[.5]:.0f} ({pe[.25]:.0f}-{pe[.75]:.0f})   [6 (3-13)]")
print(f"Days since last visit:           {rd[.5]:.0f} ({rd[.25]:.0f}-{rd[.75]:.0f})   [30 (13-82)]")

## 6 · Save

In [ ]:
save_df(P, DATA_DIR, "pred_cohort")
save_df(F, DATA_DIR, "pred_features")

counts = {"prediction_rule": PRED_RULE,
          "first_sbp_gt_140": int(len(P) + no_data_after.sum()),
          "no_data_after_prediction": int(no_data_after.sum()),
          "data_after_prediction": int(len(P))}
for H in WINDOWS:
    s = P[f"target_{H}"].dropna()
    counts[f"target_{H}"] = {"n": int(len(s)), "no_bp": int(s.sum()),
                             "data_ended": int((~P[f'covered_{H}']).sum()),
                             "died": int((P[f'covered_{H}'] & P[f'died_{H}']).sum())}
counts["competing_risk_90"] = {"bp": int((P.cr90_status == 1).sum()), "death_before_bp": int((P.cr90_status == 2).sum())}
save_json(counts, RESULTS_DIR, "03_prediction_counts")